# Week 6 Follow-up — Can We Fix the Extreme-Bucket Bias?

`notebooks/13` found the `1d` model systematically over-forecasts on demand-suppressing extremes (`is_hot`, `is_high_wind`, `is_christmas`), and that this bias directly explains why PICP collapses in exactly those buckets. Two targeted interventions, compared against the current model:

1. **`is_christmas` feature** — broader than the existing bank-holiday flags (Dec 24-Jan 1 vs. just the 1-2 actual public holidays), so it may carry information the model doesn't have yet.
2. **Sample weighting** — training rows in `is_hot`/`is_high_wind`/`is_christmas` get 3x weight, so the optimizer pays more for getting them wrong, directly countering "rare regime diluted by average loss."

Four point-model variants (A: baseline, B: +feature, C: +weighting, D: +both) isolate which lever does what. Then the most complete recipe (D) is re-run for the quantile models to check whether reduced point bias actually improves PICP — the metric that matters for the deliverable, not just a proxy for it.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from edf import config
from edf.models.baselines import previous_week_same_time
from edf.features.buckets import build_day_type_buckets, is_christmas_period
from edf.evaluate import evaluate_by_bucket, evaluate_quantiles_by_bucket
from edf.features.demand import build_feature_table
from edf.models.forecast import train_lightgbm
from edf.features.generation import capacity_factor
from edf.models.quantile import enforce_monotonic_quantiles
from edf.features.weather import build_weather_feature_table

pd.options.display.float_format = "{:.2f}".format
WEATHER_RAW_DIR = Path("../data/raw/weather")
HORIZON_PERIODS = config.HORIZONS["1d"]
POINT_PARAMS = {"num_leaves": 15, "min_child_samples": 20, "learning_rate": 0.05}
POINT_N_ESTIMATORS = 689
QUANTILE_PARAMS = {"num_leaves": 15, "min_child_samples": 50, "learning_rate": 0.1}
QUANTILE_N_ESTIMATORS = 549
UPWEIGHT = 3.0

In [2]:
df = pd.read_parquet("../data/processed/gb_energy_2020_2025.parquet")
demand = df["demand"]
train_start, train_end = config.TRAIN
val_start, val_end = config.VALIDATION

era5 = build_weather_feature_table("open_meteo_historical", df.index, raw_dir=WEATHER_RAW_DIR)

wind_cf_full = capacity_factor(df["wind"], df["wind_capacity"])
buckets_full = build_day_type_buckets(era5["temperature_c"], wind_cf_full)
buckets_val = buckets_full.loc[val_start:val_end]

problem_mask = buckets_full["is_hot"] | buckets_full["is_high_wind"] | buckets_full["is_christmas"]
sample_weight_full = pd.Series(1.0, index=df.index)
sample_weight_full[problem_mask] = UPWEIGHT
print(f"{problem_mask.mean():.1%} of all rows upweighted to {UPWEIGHT}x")

19.9% of all rows upweighted to 3.0x


## Build the two feature sets (with/without `is_christmas`)

In [3]:
era5_with_christmas = era5.copy()
era5_with_christmas["is_christmas"] = is_christmas_period(df.index)

X_plain, y = build_feature_table(df, horizon_periods=HORIZON_PERIODS, weather=era5)
X_christmas, y_c = build_feature_table(df, horizon_periods=HORIZON_PERIODS, weather=era5_with_christmas)

X_train_plain, y_train = X_plain.loc[train_start:train_end], y.loc[train_start:train_end]
X_val_plain, y_val = X_plain.loc[val_start:val_end], y.loc[val_start:val_end]
X_train_christmas, y_train_c = X_christmas.loc[train_start:train_end], y_c.loc[train_start:train_end]
X_val_christmas = X_christmas.loc[val_start:val_end]

weight_train_plain = sample_weight_full.loc[X_train_plain.index]
weight_train_christmas = sample_weight_full.loc[X_train_christmas.index]

## Train all four point-model variants

In [4]:
variants = {}

variants["A_baseline"] = train_lightgbm(X_train_plain, y_train, n_estimators=POINT_N_ESTIMATORS, **POINT_PARAMS)
variants["B_plus_feature"] = train_lightgbm(X_train_christmas, y_train_c, n_estimators=POINT_N_ESTIMATORS, **POINT_PARAMS)
variants["C_plus_weighting"] = train_lightgbm(
    X_train_plain, y_train, sample_weight=weight_train_plain, n_estimators=POINT_N_ESTIMATORS, **POINT_PARAMS
)
variants["D_plus_both"] = train_lightgbm(
    X_train_christmas, y_train_c, sample_weight=weight_train_christmas, n_estimators=POINT_N_ESTIMATORS, **POINT_PARAMS
)

X_val_by_variant = {
    "A_baseline": X_val_plain, "B_plus_feature": X_val_christmas,
    "C_plus_weighting": X_val_plain, "D_plus_both": X_val_christmas,
}
preds_by_variant = {
    name: pd.Series(model.predict(X_val_by_variant[name]), index=X_val_by_variant[name].index)
    for name, model in variants.items()
}
print("trained:", list(variants))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001305 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5691
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 29
[LightGBM] [Info] Start training from score 26931.631382


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001178 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5693
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 30
[LightGBM] [Info] Start training from score 26931.631382


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000938 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5691
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 29
[LightGBM] [Info] Start training from score 27001.358990


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001175 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5693
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 30
[LightGBM] [Info] Start training from score 27001.358990


trained: ['A_baseline', 'B_plus_feature', 'C_plus_weighting', 'D_plus_both']


## Point-model comparison: MAE / MASE / bias, per bucket

In [5]:
naive_val = previous_week_same_time(demand).loc[val_start:val_end]
bucket_names = ["all", "is_hot", "is_high_wind", "is_christmas"]

for name, preds in preds_by_variant.items():
    table = evaluate_by_bucket(y_val, preds, buckets_val, seasonal_naive_pred=naive_val)
    print(f"--- {name} ---")
    print(table.loc[bucket_names, ["mae", "mase", "bias"]])
    print()

--- A_baseline ---
                 mae  mase     bias
all           889.63  0.44  -161.26
is_hot       1205.14  0.42  -560.17
is_high_wind 1178.99  0.42  -796.00
is_christmas 1670.82  0.50 -1242.18

--- B_plus_feature ---
                 mae  mase     bias
all           884.14  0.44  -157.39
is_hot       1180.52  0.41  -524.99
is_high_wind 1155.24  0.42  -787.05
is_christmas 1459.51  0.44 -1024.84

--- C_plus_weighting ---
                 mae  mase     bias
all           890.75  0.44  -132.48
is_hot       1142.73  0.40  -501.41
is_high_wind 1125.12  0.40  -685.93
is_christmas 1631.84  0.49 -1207.91

--- D_plus_both ---
                 mae  mase     bias
all           885.70  0.44  -137.78
is_hot       1150.98  0.40  -499.17
is_high_wind 1119.38  0.40  -692.86
is_christmas 1615.68  0.49 -1192.30



## Finding: each lever fixes its own bucket, and they don't stack additively

Bias reduction (baseline → best variant), by bucket:

| bucket | baseline bias | best variant | best bias | reduction |
|---|---|---|---|---|
| `is_christmas` | -1242 | **B (+feature)** | -1025 | 17.5% |
| `is_hot` | -560 | **C (+weighting)** | -501 | 10.5% |
| `is_high_wind` | -796 | **C (+weighting)** | -686 | 13.8% |

The `is_christmas` **feature** is the more effective lever for Christmas specifically (barely moved by weighting alone: -1242 → -1208) — confirms the earlier reasoning that Christmas needed *new information* (a broader window than the existing bank-holiday flags), not just more training emphasis. **Weighting** is the more effective lever for `is_hot`/`is_high_wind` — these buckets already had the relevant continuous features (temperature, wind speed), so what they needed was the optimizer caring more about getting them right, not new inputs.

**D (both combined) is never the single best for any one bucket** — it's a compromise that's close to but not quite as good as the specialised fix in each case (e.g. `is_christmas` bias is -1192 under D vs. -1025 under B alone). Overall accuracy (`all` row) barely moves across all four variants (MAE 884-891, MASE 0.44 throughout) — none of this costs anything in the common case, which is reassuring, but also means the fixes are genuinely targeted rather than a free general improvement.

## Does reduced point bias actually improve PICP? Quantile-model comparison, baseline vs. the combined recipe (D)

Reusing `notebooks/12`'s tuned quantile hyperparameters, unchanged — only the feature set and sample weights differ between the two runs below.

In [6]:
def fit_quantiles(X_train, y_train, X_val, weight_train=None):
    preds = {}
    for alpha in [0.1, 0.5, 0.9]:
        m = train_lightgbm(
            X_train, y_train, sample_weight=weight_train, objective="quantile", alpha=alpha,
            n_estimators=QUANTILE_N_ESTIMATORS, **QUANTILE_PARAMS,
        )
        preds[alpha] = pd.Series(m.predict(X_val), index=X_val.index)
    return enforce_monotonic_quantiles(preds)

quantile_preds_baseline = fit_quantiles(X_train_plain, y_train, X_val_plain)
quantile_preds_combined = fit_quantiles(
    X_train_christmas, y_train_c, X_val_christmas, weight_train=weight_train_christmas
)

prob_baseline = evaluate_quantiles_by_bucket(y_val, quantile_preds_baseline, buckets_val)
prob_combined = evaluate_quantiles_by_bucket(y_val, quantile_preds_combined, buckets_val)
print("--- baseline (A) ---")
print(prob_baseline.loc[bucket_names])
print("\n--- combined recipe (D) ---")
print(prob_combined.loc[bucket_names])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001262 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5691
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 29
[LightGBM] [Info] Start training from score 19322.000000


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001072 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5691
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 29
[LightGBM] [Info] Start training from score 26196.000000


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001064 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5691
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 29
[LightGBM] [Info] Start training from score 36093.000000


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001068 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5693
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 30
[LightGBM] [Info] Start training from score 19437.000000


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001103 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5693
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 30
[LightGBM] [Info] Start training from score 26406.000000


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001155 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5693
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 30
[LightGBM] [Info] Start training from score 35779.000000


--- baseline (A) ---
              pinball_0.1  pinball_0.5  pinball_0.9  picp        n
all                239.77       447.38       200.30  0.65 17568.00
is_hot             368.05       607.83       274.68  0.56  1615.00
is_high_wind       356.33       588.51       231.10  0.61  1867.00
is_christmas       738.27       847.79       304.27  0.43   432.00

--- combined recipe (D) ---
              pinball_0.1  pinball_0.5  pinball_0.9  picp        n
all                233.65       446.88       201.84  0.67 17568.00
is_hot             349.33       580.39       265.95  0.53  1615.00
is_high_wind       344.66       559.32       220.58  0.58  1867.00
is_christmas       658.30       809.34       259.68  0.38   432.00


## Finding: pinball loss improved everywhere — but PICP got *worse* in exactly the buckets we targeted

| bucket | PICP (A) | PICP (D) | change |
|---|---|---|---|
| all | 0.652 | **0.666** | improved |
| `is_hot` | 0.558 | 0.529 | **worse** |
| `is_high_wind` | 0.609 | 0.583 | **worse** |
| `is_christmas` | 0.426 | 0.384 | **worse** |

This is the opposite of what the point-model result would suggest. Pinball loss (what the quantile models actually optimize) improved in every single bucket under D, including the three targeted ones — so by the training objective's own yardstick, D's quantiles are unambiguously "better." But PICP — actual empirical coverage, the metric that matters for the deliverable — got *worse* specifically where we were trying to fix it.

**Why these can point in different directions**: pinball loss rewards being *closer* to the true quantile value; PICP only counts whether the actual fell *inside* the interval at all. Reducing bias shifts the whole [P10, P90] band toward the true distribution's centre — genuinely closer on average, lower pinball loss — but if the band's *width* doesn't widen correspondingly (or narrows, e.g. because upweighting a smaller set of extreme rows nudges the model toward being more confident about them, having now "seen" them more), a tighter-but-still-imperfectly-centred interval can cover the truth *less* often even while sitting closer to it on average. This isn't a bug in the experiment — it's a genuine, instructive illustration of why Week 5 evaluates both pinball loss *and* PICP rather than either alone: they measure different things, and optimizing one doesn't guarantee the other. **Bottom line: this recipe is worth keeping for the point model (real, if modest, bias reduction with no cost elsewhere) but does not fix the probabilistic model's calibration problem** — that remains open, and now with sharper evidence that a feature/weighting fix targeting bias isn't the right lever for it. The stretch goal already in `PLAN.md` (a genuinely distributional model, e.g. via conformal prediction) is looking more clearly like the right next step for calibration specifically, separate from this bias-correction work.

## Isolating the cause: full 4-way decomposition for the quantile models

The comparison above only tested baseline (A) vs. the combined recipe (D), so it couldn't say whether the PICP regression came from the `is_christmas` feature, the sample weighting, or their interaction. Running the same A/B/C/D split used for the point model, but for the quantile models, isolates it.

In [7]:
quantile_variants = {
    "A_baseline": quantile_preds_baseline,  # already computed above
    "B_plus_feature": fit_quantiles(X_train_christmas, y_train_c, X_val_christmas),
    "C_plus_weighting": fit_quantiles(X_train_plain, y_train, X_val_plain, weight_train=weight_train_plain),
    "D_plus_both": quantile_preds_combined,  # already computed above
}

picp_by_variant = pd.DataFrame(
    {
        name: evaluate_quantiles_by_bucket(y_val, preds, buckets_val).loc[bucket_names, "picp"]
        for name, preds in quantile_variants.items()
    }
)
picp_by_variant

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001163 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5693
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 30
[LightGBM] [Info] Start training from score 19322.000000


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001113 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5693
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 30
[LightGBM] [Info] Start training from score 26196.000000


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001130 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5693
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 30
[LightGBM] [Info] Start training from score 36093.000000


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001097 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5691
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 29
[LightGBM] [Info] Start training from score 19437.000000


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000999 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5691
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 29
[LightGBM] [Info] Start training from score 26406.000000


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001072 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5691
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 29
[LightGBM] [Info] Start training from score 35779.000000


,A_baseline,B_plus_feature,C_plus_weighting,D_plus_both
all,0.65,0.65,0.66,0.67
is_hot,0.56,0.56,0.52,0.53
is_high_wind,0.61,0.59,0.57,0.58
is_christmas,0.43,0.41,0.41,0.38


## Finding: sample weighting is the dominant cause — and the decision

| bucket | A (baseline) | B (+feature) | C (+weighting) | D (+both) |
|---|---|---|---|---|
| `is_hot` | 0.558 | 0.562 | **0.515** | 0.529 |
| `is_high_wind` | 0.609 | 0.590 | **0.570** | 0.583 |
| `is_christmas` | 0.426 | 0.414 | 0.414 | **0.384** |
| all | 0.652 | 0.646 | 0.664 | 0.666 |

**Sample weighting (C) is the dominant cause of the PICP drop in `is_hot`/`is_high_wind`** — it alone accounts for a larger regression than the feature does (-0.043 vs. +0.004 for `is_hot`; -0.039 vs. -0.019 for `is_high_wind`). The `is_christmas` **feature** independently hurts `is_christmas` PICP too (0.426→0.414), even though it's the lever that *helped* Christmas's point bias — so this isn't purely a weighting problem. Combining both (D) isn't additive: it's the worst outcome for `is_christmas` (0.384) but sits *between* B and C alone for the other two buckets, suggesting some dilution when both act together. Both interventions *improve* overall PICP (0.652→0.664-0.666) — calibration is being redistributed away from the extreme buckets toward the majority of rows, the opposite of what we wanted.

**Mechanism**: pushing the optimizer to fit these rows' *point* value more precisely (lower pinball loss, reduced bias) plausibly also makes the model more "confident" — narrower — about them specifically, since it's now seeing them (or an amplified version of them) more often during training. A narrower interval covers the truth less often even when it's centred better on average.

### Decision

**Keep both interventions for the point model** (bias reductions of 10-18% in the targeted buckets, no cost to overall accuracy — an unambiguous win there). **Do not apply either to the quantile/probabilistic model.** Both make calibration measurably worse in exactly the buckets — extreme events — where good calibration matters most for real decision-making, even though they improve the training objective (pinball loss) and the aggregate PICP number. This is a case where the metric that's easy to optimize (pinball loss) and the metric that actually matters (bucket-level PICP) point in different directions, and the latter wins the decision. The probabilistic model's tail-calibration problem stays open for a different kind of fix — the `PLAN.md` stretch goal (a genuinely distributional model / conformal prediction) rather than a bias-correction recipe borrowed from the point model.